# 2025-07-10: Longitudinal MM Metadata Design
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology

**Main aim**: \
This notebook filters and prepares metadata and file information for FH1 BMMC, FH1 PBMC, BRI PBMC, and MM Healthy samples from HISE. It merges in manual annotations (e.g., treatment, response) and lab values, ensuring a clean, deduplicated dataset for downstream analysis. The metadata for the NDMM study has been manually cleaned by the scientific team. I will be cleaning up and formatting the metadata to match the needs of the scRNAseq analysis efforts.

In [1]:
# install.packages('lubridate')
# install.packages('dplyr')

suppressPackageStartupMessages({
    library(plyr)
    library(dplyr) 
    library(hise)
    library(lubridate)
    library(tidyr)
    library(stringr)
})

Warning message:
“package ‘plyr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘stringr’ was built under R version 4.4.3”


### 1. Get sample descriptors from HISE

In [2]:
# Access descriptors data from HISE
desc <- getFileDescriptors(fileType = "scRNA-seq-labeled", toDF=TRUE)

[2026-03-25 13:33:05] INFO  Calling getFileDescriptors


New names:
• `datahistory_record_id_oldvalue` -> `datahistory_record_id_oldvalue...577`
• `datahistory_record_id_newvalue` -> `datahistory_record_id_newvalue...578`
• `record_id` -> `record_id...815`
• `datahistory_record_id_oldvalue` -> `datahistory_record_id_oldvalue...816`
• `datahistory_record_id_newvalue` -> `datahistory_record_id_newvalue...817`
• `record_id` -> `record_id...827`


[2026-03-25 13:39:30] INFO  Finished getFileDescriptors (success=TRUE, time_elapsed=382.545s)


In [3]:
desc_table <- desc$descriptors
idx <- match(desc_table$sample.id, desc$specimens$sampleId)
desc_table$specimen.specimenGuid   <- desc$specimens$specimenGuid[idx]
desc_table$specimen.specimenType   <- desc$specimens$specimenType[idx]
desc_table$specimen.specimenStatus <- desc$specimens$specimenStatus[idx]

### 2. Define manual cases for visit subsets and healthy BMMC UUIDs

In [4]:
# Define reference sets
selected_visits <- c(
  "Flu Year 1 Stand-Alone", "Flu Year 1 Day 0", "Flu Year 1 Day 7", "Flu Year 1 Day 90",
  "Flu Year 2 Stand-Alone", "Flu Year 2 Day 0", "Flu Year 2 Day 7", "Flu Year 2 Day 90"
)

healthy_uuids <- c(
  "5f536304-7f6b-4c4f-bff5-4b0e11067c14", "b5b5b835-9509-4ab7-99a8-87c6bd972b4d",
  "70606098-48a1-4e2c-8168-22ece96150e4", "c142c6ff-49ca-40e8-8f48-15f7f8b1f32b",
  "f3eff137-c48b-40c7-b91d-434347003c7a", "6557f5d6-a686-4bb8-8c5a-862ff1728c8c",
  "d44558a4-3f7f-4c5c-b8c3-8c513ac7c942", "8b70332c-29bd-4079-862d-559d189c692c",
  "f372defd-afec-450a-b48c-7976e9141619", "fa185ac8-5ad1-4499-a3e4-6fb69dc9331f"
)

# Import manually matched BRI samples (CSV available on github)
bri_uuid <- read.csv("../../../data/rna/metadata/Sample_match_FH1_BRI.csv")

In [5]:
# Subset relevant samples 
scrna_meta <- subset(desc_table, cohort.cohortGuid == "FH1")

desc_bri_subset <- desc_table %>%
  filter(
    subject.subjectGuid %in% bri_uuid$Subject[bri_uuid$cohort == "BRI"],
    sample.visitName %in% selected_visits
  ) %>%
  mutate(sample.visitName = factor(sample.visitName, levels = selected_visits))
  # Note: factor ordering is lost after bind_rows() — reapply if needed downstream

desc_healthy_subset <- desc_table %>%
  filter(file.id %in% healthy_uuids)

### 3. Combine BRI, FH1 and healthy into 1 table

In [6]:
# Combine subsets
scrna_meta <- bind_rows(scrna_meta, desc_bri_subset, desc_healthy_subset)

### 4. Add in manually defined fields
> Fields are defined for Tissue, Analysis Category
> Pulling out the `pbmc_sample_id` field to match that of the HISE scRNA metadata file format. This is not intuitive on the first run but once you've checked out the files it makes more sense. 

In [7]:
scrna_meta$pbmc_sample_id <- str_extract(scrna_meta$file.name, "(PB|BMC|SP)[0-9]+-[0-9]+")

# Derive tissue type from the Specimen ID
# PB=peripheral blood, PL=plasma, CT/BL/TP=other blood draws → PBMC
# BM=bone marrow, SP=spleen → BMMC
scrna_meta <- scrna_meta %>%
  mutate(tissue = case_when(
    grepl("PB|PL|CT|BL|TP", pbmc_sample_id) ~ "PBMC",
    grepl("BM|SP", pbmc_sample_id) ~ "BMMC",
    TRUE ~ pbmc_sample_id
  ))

In [8]:
# Categorize samples for downstream analysis
scrna_meta <- scrna_meta %>%
  mutate(
    manual.category = case_when(
      subject.subjectGuid %in% bri_uuid$Subject[bri_uuid$cohort == "BRI"] ~ "healthy_pbmc",
      file.id %in% healthy_uuids ~ "healthy_bmmc",
      cohort.cohortGuid == "FH1" & tissue == "PBMC" ~ "tumor_pbmc",
      cohort.cohortGuid == "FH1" & tissue == "BMMC" ~ "tumor_bmmc",
      TRUE ~ NA_character_
    )
  )

In [9]:
#  Add in data for treatment and flu response 
dara_ids <- c("FH1019", "FH1020", "FH1022", "FH1023",
          "FH1024", "FH1026", "FH1027", "FH1028")

flu_non_responders <- c("FH1021", "FH1016", "FH1011", "FH1009", "FH1007", "FH1004", "FH1003")
flu_responders     <- c("FH1002", "FH1005", "FH1006", "FH1008", "FH1012", "FH1014", "FH1017")

scrna_meta <- scrna_meta %>%
  mutate(
    manual.treatment_dara = if_else(subject.subjectGuid %in% dara_ids, "dara", "non_dara"),
    manual.flu_response = case_when(
      subject.subjectGuid %in% flu_non_responders ~ "flu_non_responder",
      subject.subjectGuid %in% flu_responders ~ "flu_responder",
      TRUE ~ NA_character_
    )
  )

### 5. Clean up the dataframe

In [10]:
# Row filters must come before column cleaning to avoid dropping key columns (e.g. sample.sampleKitGuid)
# KT01500 is a known failed kit; EXP- batches are experimental/non-production runs
scrna_meta <- scrna_meta %>%
  dplyr::filter(
    sample.sampleKitGuid != "KT01500",
    !startsWith(file.batchID, "EXP-")
  ) %>%
  select(where(~ !all(is.na(.) | as.character(.) %in% c("NA", ""))))

In [11]:
# Set disease state
scrna_meta <- scrna_meta %>%
  mutate(sample.diseaseStatesRecordedAtVisit = case_when(
    file.id %in% healthy_uuids |
    subject.subjectGuid %in% bri_uuid$Subject[bri_uuid$cohort == "BRI"] ~ "Healthy",
    manual.category %in% c("tumor_pbmc", "tumor_bmmc") ~ "Multiple Myeloma",
    TRUE ~ sample.diseaseStatesRecordedAtVisit
  ))

In [12]:
# Normalize missing values
scrna_meta <- scrna_meta %>%
  mutate(
    sample.daysSinceFirstVisit = ifelse(
      (file.id %in% healthy_uuids |
       subject.subjectGuid %in% bri_uuid$Subject[bri_uuid$cohort == "BRI"]) &
      sample.daysSinceFirstVisit == "", 0, sample.daysSinceFirstVisit
    ),
    sample.daysSinceFirstVisit = as.numeric(sample.daysSinceFirstVisit)
  )

# Replace empty strings with NA in character columns only (avoids coercion errors on numeric/Date cols)
char_cols <- names(scrna_meta)[sapply(scrna_meta, is.character)]
scrna_meta[, (char_cols) := lapply(.SD, function(x) dplyr::na_if(x, "")), .SDcols = char_cols]

In [13]:
# Set Healthy label for visitName and visitDetails
scrna_meta <- scrna_meta %>%
  mutate(
    sample.visitName = ifelse(manual.category == "healthy_bmmc", "Healthy", sample.visitName),
    sample.visitDetails = ifelse(manual.category == "healthy_bmmc", "Healthy", sample.visitDetails)
  )

### 6. Delete duplicates based on sepcimen ID and time stamp
> Specimen ID is the most unique data field and the time stamp is extracted from the file name. Here, we're removing the old samples and only keeping the new samples. 

In [14]:
# Extract timestamp from filename
# File path format: .../YYYY-MM-DDTHH:MM:SS.../... — timestamp is always the 3rd path element
add_time_stamp <- function(df) {
  df$manual.time_stamp <- sapply(strsplit(df$file.name, "/"), function(x) x[3])
  df
}

# Some kits have multiple processed versions; keep the most recent by timestamp
# Remove all duplicates based on timestamp
scrna_meta <- scrna_meta %>%
  add_time_stamp() %>%
  mutate(manual.time_stamp = as.Date(manual.time_stamp)) %>%
  arrange(sample.sampleKitGuid, desc(manual.time_stamp)) %>%
  distinct(sample.sampleKitGuid, .keep_all = TRUE)

### 7. Add in manually curated metadata for FH1 cohort
> The FH1 cohort has a bunch of manually curated data in a spreadsheet. Here, we're adding that data in. Note that for the other cohorts, all of these values will end up being NA values. 

In [15]:
# Also grab the manually curated spreadsheet -- Supplemental table 1 -- and update to match scRNA column name conventions
manual_metadata <- read.csv('../../../manuscript-figures/inputs/metadata/sup_table1.csv')

manual_metadata <- manual_metadata %>%
  dplyr::rename(
    subject.subjectGuid            = Subject,
    sample.visitDetails            = visitDetails,
    sample.sampleKitGuid           = sampleKitGuid,  
    subject.cmv                    = CMV.Ab.Screen.Result,
    cmv.ab_screen_index_value      = CMV.Ab.Screen.Index.Value,
    igg.iga                        = Immunoglobulin.A,
    igg.igg                        = Immunoglobulin.G,
    igg.igm                        = Immunoglobulin.M,
    igg.immunofixation             = Immunofixation,
    igg.k_flc                      = Kappa.Free.Light.Chain,
    igg.kl_flc_ratio               = Kappa.Lambda.FLC.Ratio,
    igg.l_flc                      = Lambda.Free.light.chain,
    igg.mono_1_id                  = Monoclonal.1.ID..Serum,
    igg.mono_1_quant               = Monoclonal.1.Quantification..Serum,
    lip.cholesterol_hdl            = Cholesterol..HDL,
    lip.cholesterol_ldl            = Cholesterol..LDL,
    lip.cholesterol_non_hdl        = Cholesterol..Non.HDL,
    lip.cholesterol_total          = Cholesterol..Total,
    lip.triglycerides              = Triglycerides,
    cyto.17p_loss                  = X17p.,
    cyto.1p_loss                   = X1p.,
    cyto.high_risk                 = High.risk.cytogenetics...17p..t.4.14..,
    cyto.karyotype                 = Karyotype,
    cyto.t11_14                    = t.11.14.,
    cyto.t14_16                    = t.14.16.,
    cyto.t14_20                    = t.14.20.,
    cyto.t4_14                     = t.4.14.,
    tb.end_induction_response = Investigator.assessed.response.category..based.on.available.data.
  )

In [16]:
# Define field groups
lab_fields <- c(
  "subject.cmv", "cmv.ab_screen_index_value", "cmv.ab_visit", 
  "igg.iga", "igg.igg", "igg.igm", "igg.immunofixation", "igg.k_flc", 
  "igg.kl_flc_ratio", "igg.l_flc", "igg.mono_1_id", "igg.mono_1_quant",
  "lip.cholesterol_hdl_ratio", "lip.cholesterol_hdl", "lip.cholesterol_ldl", 
  "lip.cholesterol_non_hdl", "lip.cholesterol_total", "lip.triglycerides",
  "RISS.stage", "cyto.17p_loss", "cyto.1q_gain", "cyto.1p_loss", "cyto.high_risk",
  "cyto.karyotype", "cyto.t11_14", "cyto.t14_16", "cyto.t14_20", "cyto.t4_14"
)

# Get lab subset and fill NAs
subset_labs <- manual_metadata %>%
  select(subject.subjectGuid, sample.sampleKitGuid, sample.visitDetails, all_of(lab_fields)) %>%
  mutate(across(everything(), ~replace_na(as.character(.x), "Not Recorded")))

> Now, response mapping is a bit harder because its Subject x Time point specific. A persons response to treatment can look different over their course of treatment. 

In [17]:
# Get response subset and fill NAs
subset_response <- manual_metadata %>%
  select(subject.subjectGuid, 
         sample.sampleKitGuid, 
         sample.visitDetails,
         tb.end_induction_response,
         tb.post_trans_90d_response,
         tb.PostTX_1Yr_response,
         tb.PostTX_2Yr_response) %>%
  mutate(across(everything(), ~replace_na(as.character(.x), "Not Recorded")))

# Reshape and filter response data
df <- subset_response %>%
  pivot_longer(cols = starts_with("tb."),
               names_to = "ResponseType",
               values_to = "Response") %>%
  filter(Response != "Not Recorded") %>%
  arrange(subject.subjectGuid)

# Create mapping vectors
# Note: setNames() keeps only the last entry if a sampleKitGuid appears multiple times across timepoints
response_map <- setNames(df$Response, df$sample.sampleKitGuid)
response_type_map <- setNames(df$ResponseType, df$sample.sampleKitGuid)

# Apply mappings to scrna_meta
scrna_meta <- scrna_meta %>%
  mutate(
    manual.response = response_map[sample.sampleKitGuid],
    manual.response_type = response_type_map[sample.sampleKitGuid]
  )

In [18]:
scrna_meta <- scrna_meta %>%
  left_join(subset_labs, by = "sample.sampleKitGuid")

scrna_meta <- scrna_meta %>%
  distinct(`sample.sampleKitGuid`, .keep_all = TRUE)

In [19]:
# left_join creates duplicate cols with .x/.y suffixes when col names overlap between scrna_meta and subset_labs
# Keep the .x (scrna_meta) version and strip the suffix
cols <- colnames(scrna_meta)
base_names <- gsub("\\.x$|\\.y$", "", cols)
dedup_cols <- !duplicated(base_names)
scrna_meta <- scrna_meta[, dedup_cols, with = FALSE]
colnames(scrna_meta) <- gsub("\\.x$|\\.y$", "", colnames(scrna_meta))

### 8. Save output

In [21]:
dim(scrna_meta)

[1] 486  71

In [22]:
head(scrna_meta)

projectGuid.projectGuid,mergeKey.mergeKey,cohort.cohortGuid,file.id,file.name,file.batchID,file.pool,file.fileType,file.majorVersion,labLastModified.labLastModified,⋯,RISS.stage,cyto.17p_loss,cyto.1q_gain,cyto.1p_loss,cyto.high_risk,cyto.karyotype,cyto.t11_14,cyto.t14_16,cyto.t14_20,cyto.t4_14
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
e206cf7a-5b13-478f-b842-a305fe4954d8,119c13a2-9d5a-4c39-89f4-8c7997d30903_c9be600f-50b8-48cf-9d8a-b4d84b0eba6f,BR2,119c13a2-9d5a-4c39-89f4-8c7997d30903,automated/merged/2023-11-17T21:36:51.794326181Z/Version-1/42103e6b-6798-4798-b735-fd620f40847b/B002/labeled/B002-P1_PB00015-01_2023-11-17T21:36:51.794326181Z_labeled.h5,B002,NA,scRNA-seq-labeled,1,2021-09-13T20:09:06.402Z,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
e206cf7a-5b13-478f-b842-a305fe4954d8,40ded1aa-c1a0-44aa-9920-85da60436101_24a96765-0909-47d6-b95a-8dd90a4b0267,BR2,40ded1aa-c1a0-44aa-9920-85da60436101,automated/merged/2023-11-17T21:38:04.103392546Z/Version-1/42103e6b-6798-4798-b735-fd620f40847b/B002/labeled/B002-P2_PB00020-01_2023-11-17T21:38:04.103392546Z_labeled.h5,B002,NA,scRNA-seq-labeled,1,2021-09-13T20:09:06.633Z,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
e206cf7a-5b13-478f-b842-a305fe4954d8,247d580e-7b47-4f59-bc37-6e29188b7aff_fc01c04c-7213-471f-8510-1785f9dee039,BR1,247d580e-7b47-4f59-bc37-6e29188b7aff,automated/merged/2021-08-19T17:09:29.934849811Z/Version-1/42103e6b-6798-4798-b735-fd620f40847b/B078/labeled/B078-P2_PB00025-04_2021-08-19T17:09:29.934849811Z_labeled.h5,B078,NA,scRNA-seq-labeled,1,2021-09-13T20:09:06.872Z,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
e206cf7a-5b13-478f-b842-a305fe4954d8,4d075c3d-1fb6-4464-a8d3-68987b638100_c4a57fdf-be4b-4910-b62a-6e753e71643b,BR1,4d075c3d-1fb6-4464-a8d3-68987b638100,automated/merged/2021-08-19T17:09:29.934849811Z/Version-1/42103e6b-6798-4798-b735-fd620f40847b/B078/labeled/B078-P2_PB00029-02_2021-08-19T17:09:29.934849811Z_labeled.h5,B078,NA,scRNA-seq-labeled,1,2021-09-13T20:09:06.951Z,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
e206cf7a-5b13-478f-b842-a305fe4954d8,8f472610-4c00-4daa-bba6-96448a82fffc_990f590a-83ab-4c31-95bc-f8bec685e96a,FH1,8f472610-4c00-4daa-bba6-96448a82fffc,automated/merged/2021-06-15T23:04:55.760038239Z/B017/labeled/B017-P1_PB00132-01_2021-06-15T23:04:55.760038239Z_labeled.h5,B017,NA,scRNA-seq-labeled,NA,2022-06-13T20:50:34.627Z,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
e206cf7a-5b13-478f-b842-a305fe4954d8,09e1c4ff-2a64-4bf5-881b-3c1a46fc2c35_ea076b22-4a7c-49fa-9fbc-d38300386cdb,FH1,09e1c4ff-2a64-4bf5-881b-3c1a46fc2c35,automated/merged/2021-11-25T03:04:54.542561181Z/Version-1/42103e6b-6798-4798-b735-fd620f40847b/B095/labeled/B095-P2_PB00133-01_2021-11-25T03:04:54.542561181Z_labeled.h5,B095,NA,scRNA-seq-labeled,1,2024-01-05T22:46:14.052Z,⋯,II,No,Yes,No,No,Normal,No,No,Not Tested,No


In [24]:
# Final output includes FH1, BRI, and healthy BMMC samples with manual annotations merged in
# Non-FH1 samples will have NA for all manual lab/response fields
output_path <- '../../../data/rna/metadata'
write.csv(scrna_meta, file.path(output_path, 'scrna_metadata.csv'))